# Практика 2. Движение в видео: оптический поток, фон и стабилизация

## 0. Импорты и вспомогательные функции

Достаточно `numpy`, `opencv-python` и `matplotlib`. Ядро Python 3.10+ подходит.

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

In [ ]:
def show(images, titles, cmaps=None, figsize=(15, 4)):
    'Show images in one row with axes hidden.'
    if cmaps is None:
        cmaps = [None] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for ax, image, title, cmap in zip(axes, images, titles, cmaps):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def flow_to_rgb(flow, clip=8.0):
    'Encode direction as hue and magnitude as brightness.'
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0] = np.uint8(angle * 90 / np.pi)
    hsv[..., 1] = 255
    hsv[..., 2] = np.uint8(255 * np.clip(magnitude / clip, 0, 1))
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def endpoint_error(flow, gt, valid=None):
    'Mean Euclidean error between predicted and ground-truth flow.'
    error = np.linalg.norm(flow.astype(np.float32) - gt.astype(np.float32), axis=-1)
    if valid is None:
        valid = np.ones(error.shape, dtype=bool)
    return float(error[valid].mean())


def f1_score(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))

## 1. Синтетическая пара с известным движением

Камера сдвигается на `(3, -2)` px. Текстурный объект дополнительно движется на
`(4, 3)` px относительно сцены. Поэтому истинный поток равен `(3, -2)` на фоне
и `(7, 1)` на объекте. На границах возникают окклюзии, поэтому метрику считаем
внутри изображения и отдельно по фону/объекту.

### От постоянства яркости к Lucas–Kanade

Пусть одна и та же точка сцены сохраняет яркость:
$I(x, y, t)=I(x+u, y+v, t+1)$. Линеаризация правой части даёт
$I_xu+I_yv+I_t=0$. Одного уравнения недостаточно для двух компонент $(u,v)$,
поэтому Lucas–Kanade предполагает одинаковое движение в окне $W$ и решает МНК:

$$
\begin{bmatrix}
\sum_W I_x^2 & \sum_W I_xI_y\\
\sum_W I_xI_y & \sum_W I_y^2
\end{bmatrix}
\begin{bmatrix}u\\v\end{bmatrix}
=-
\begin{bmatrix}\sum_W I_xI_t\\\sum_W I_yI_t\end{bmatrix}.
$$

Левая матрица — структурный тензор. Два достаточно больших собственных значения
означают, что движение наблюдаемо в двух направлениях. Малое $\lambda_{min}$
указывает на однородную область или край и делает решение неустойчивым.

In [ ]:
def shift_image(image, dx, dy, border_mode=cv2.BORDER_REFLECT):
    matrix = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(image, matrix, (image.shape[1], image.shape[0]),
                          flags=cv2.INTER_LINEAR, borderMode=border_mode)


def make_texture(height=180, width=240, seed=7):
    rng = np.random.default_rng(seed)
    noise = rng.normal(127, 42, (height, width)).astype(np.float32)
    texture = cv2.GaussianBlur(noise, (0, 0), 1.2)
    for x in range(15, width, 35):
        cv2.line(texture, (x, 8), (x, height - 8), 35 + x % 120, 1)
    for y in range(20, height, 40):
        cv2.circle(texture, (width // 3, y), 7, 220, 2)
    return np.uint8(np.clip(texture, 0, 255))


def make_flow_pair():
    height, width = 180, 240
    camera = np.array([3.0, -2.0], dtype=np.float32)
    object_relative = np.array([4.0, 3.0], dtype=np.float32)
    background = make_texture(height, width)

    y0, y1, x0, x1 = 68, 116, 82, 142
    yy, xx = np.mgrid[0:y1-y0, 0:x1-x0]
    patch = np.uint8(45 + 190 * ((xx // 6 + yy // 6) % 2))

    frame1 = background.copy()
    frame1[y0:y1, x0:x1] = patch
    frame2 = shift_image(background, *camera)
    nx0 = int(x0 + camera[0] + object_relative[0])
    ny0 = int(y0 + camera[1] + object_relative[1])
    frame2[ny0:ny0 + patch.shape[0], nx0:nx0 + patch.shape[1]] = patch

    object_mask = np.zeros((height, width), dtype=bool)
    object_mask[y0:y1, x0:x1] = True
    gt = np.zeros((height, width, 2), dtype=np.float32)
    gt[...] = camera
    gt[object_mask] = camera + object_relative
    valid = np.zeros((height, width), dtype=bool)
    valid[10:-10, 10:-10] = True
    return frame1, frame2, gt, object_mask, valid


frame1, frame2, gt_flow, object_mask, valid_mask = make_flow_pair()
show([frame1, frame2, flow_to_rgb(gt_flow)],
     ["Кадр t", "Кадр t+1", "Истинный поток (HSV)"],
     ["gray", "gray", None])

## 2. Разреженный поток: Shi–Tomasi + KLT

Из уравнения постоянства яркости получается одно ограничение
$I_xu + I_yv + I_t = 0$ на две неизвестные. Lucas–Kanade собирает ограничения
в окне, а Shi–Tomasi выбирает точки, где оба собственных значения структурного
тензора достаточно велики.

**Задание 1** Реализуйте `track_points`:

1. найдите до 120 углов `cv2.goodFeaturesToTrack`;
2. передайте их в `cv2.calcOpticalFlowPyrLK`;
3. оставьте точки со статусом `1`;
4. верните старые точки, новые точки и их смещения.

In [ ]:
def track_points(prev, nxt, max_corners=120):
    # TODO: goodFeaturesToTrack -> calcOpticalFlowPyrLK -> фильтрация status == 1
    # Рекомендуемые параметры Shi–Tomasi:
    # qualityLevel=0.01, minDistance=8, blockSize=7
    # Рекомендуемые параметры KLT:
    # winSize=(21, 21), maxLevel=3
    points0 = cv2.goodFeaturesToTrack(
        prev, maxCorners=max_corners, qualityLevel=0.01, minDistance=8, blockSize=7
    )
    if points0 is None:
      return np.empty((0,2)), np.empty((0,2)), np.empty((0,2))
    points1, status, _ = cv2.calcOpticalFlowPyrLK(
        prev, nxt, points0, None, winSize=(21, 21), maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
    )
    good = status.ravel() == 1
    old = points0.reshape(-1, 2)[good]
    new = points1.reshape(-1, 2)[good]
    # print(points0)
    return old, new, new - old

In [ ]:
# old_pts, new_pts, sparse_flow = track_points(frame1, frame2)


In [ ]:
old_pts, new_pts, sparse_flow = track_points(frame1, frame2)
point_x = np.clip(np.rint(old_pts[:, 0]).astype(int), 0, frame1.shape[1] - 1)
point_y = np.clip(np.rint(old_pts[:, 1]).astype(int), 0, frame1.shape[0] - 1)
point_is_object = object_mask[point_y, point_x]

plt.figure(figsize=(9, 5))
plt.imshow(frame1, cmap="gray")
plt.quiver(old_pts[:, 0], old_pts[:, 1], sparse_flow[:, 0], sparse_flow[:, 1],
           color=np.where(point_is_object, "tomato", "cyan"),
           angles="xy", scale_units="xy", scale=1, width=0.004)
plt.title("KLT: голубой — фон, красный — объект")
plt.axis("off")
plt.show()

for name, select, truth in [
    ("фон", ~point_is_object, np.array([3, -2])),
    ("объект", point_is_object, np.array([7, 1])),
]:
    if select.any():
        median = np.median(sparse_flow[select], axis=0)
        print(f"{name:7s}: точек={select.sum():3d}, median={median.round(2)}, GT={truth}")

### Апертурная проблема в одной карте

`cornerMinEigenVal` возвращает меньшее собственное значение структурного тензора.
На углу оно велико; на прямом крае одно направление почти не наблюдается, поэтому
$\lambda_{{min}}$ мало. Сравните карту с положениями точек KLT.

In [ ]:
lambda_min = cv2.cornerMinEigenVal(frame1, blockSize=7, ksize=3)
plt.figure(figsize=(9, 5))
plt.imshow(lambda_min, cmap="magma")
plt.scatter(old_pts[:, 0], old_pts[:, 1], s=14, facecolors="none", edgecolors="cyan")
plt.colorbar(label=r"$\lambda_{min}$")
plt.title(r"Shi–Tomasi выбирает локальные максимумы $\lambda_{min}$")
plt.axis("off")
plt.show()

selected_values = lambda_min[point_y, point_x]
print("Медиана lambda_min по изображению:", float(np.median(lambda_min)))
print("Медиана lambda_min в KLT-точках:", float(np.median(selected_values)))

Почему увеличение окна KLT иногда помогает, но может
ухудшить оценку рядом с границей движущегося объекта?

## 3. Плотный поток Farnebäck

Farnebäck аппроксимирует окрестности квадратичными полиномами и оценивает смещение
их коэффициентов на пирамиде. В отличие от KLT, результат задаётся в каждом пикселе.

**Задание 2** Реализуйте обёртку с параметрами
`pyr_scale=0.5, levels=3, winsize=21, iterations=3, poly_n=5, poly_sigma=1.2`.
Замерьте только сам вызов OpenCV и верните поле `float32 [H, W, 2]` и миллисекунды.

In [ ]:
def farneback(prev, nxt, winsize=21, levels=3):
    # TODO: time.perf_counter + cv2.calcOpticalFlowFarneback
    start = time.perf_counter()
    flow = cv2.calcOpticalFlowFarneback(
        prev, nxt, None, pyr_scale=0.5, levels=levels, winsize=winsize,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )
    elapsed_ms = (time.perf_counter() - start) * 1000
    return flow.astype(np.float32), elapsed_ms

In [ ]:
dense_flow, elapsed_ms = farneback(frame1, frame2)
background_valid = valid_mask & ~object_mask
object_valid = valid_mask & object_mask

print(f"Время: {elapsed_ms:.1f} мс")
print(f"EPE весь кадр: {endpoint_error(dense_flow, gt_flow, valid_mask):.3f} px")
print(f"EPE фон:       {endpoint_error(dense_flow, gt_flow, background_valid):.3f} px")
print(f"EPE объект:    {endpoint_error(dense_flow, gt_flow, object_valid):.3f} px")

error_map = np.linalg.norm(dense_flow - gt_flow, axis=-1)
show([flow_to_rgb(gt_flow), flow_to_rgb(dense_flow), error_map],
     ["Истинный поток", "Farnebäck", "EPE по пикселям"],
     [None, None, "inferno"])

**Мини-эксперимент** Запустите `winsize` из `{9, 21, 41}` и заполните
таблицу: время, EPE фона, EPE объекта. Объясните компромисс между подавлением шума
и размыванием границы движения.

In [ ]:
# TODO: переберите winsize = 9, 21, 41 и выведите 4 столбца:
# winsize, время, EPE фона, EPE объекта.
row = []
for window in (9, 21, 41):
  dense_flow, elapsed_ms = farneback(frame1, frame2, winsize=window)
  row.append([window, elapsed_ms, endpoint_error(dense_flow, gt_flow, background_valid), endpoint_error(dense_flow, gt_flow, object_valid)])
print(row)

### Большое смещение и пирамида

Линеаризация потока локальна: большое смещение нарушает предположение «малого шага».
На каждом уровне пирамиды изображение уменьшается вдвое, поэтому смещение 14 px
становится 7, 3.5, … px. Сравните Farnebäck без пирамиды и с четырьмя уровнями.

In [ ]:
large_dx = 14.0
large_frame2 = shift_image(frame1, large_dx, 0.0)
large_gt = np.zeros_like(gt_flow)
large_gt[..., 0] = large_dx
large_valid = np.zeros(frame1.shape, dtype=bool)
large_valid[20:-20, 20:-20] = True

large_results = []
for level_count in (1, 4):
    estimate, ms = farneback(frame1, large_frame2, winsize=21, levels=level_count)
    error = endpoint_error(estimate, large_gt, large_valid)
    large_results.append((level_count, estimate, ms, error))
    print(f"levels={level_count}: EPE={error:.3f} px, время={ms:.1f} мс")

show(
    [frame1, large_frame2, flow_to_rgb(large_results[0][1], clip=14),
     flow_to_rgb(large_results[1][1], clip=14)],
    ["Кадр t", "Сдвиг 14 px", "levels=1", "levels=4"],
    ["gray", "gray", None, None], figsize=(16, 4),
)

 На сколько пикселей уменьшается исходное смещение на самом грубом
уровне при `levels=4`? Почему слишком много уровней тоже может повредить тонким деталям?

## 4. Вычитание фона MOG2

Теперь камера неподвижна, а прямоугольник движется. У него есть серая «тень».
MOG2 кодирует фон как `0`, тень как `127`, уверенный передний план как `255`.
Первые 12 кадров пусты и нужны для прогрева модели.

In [ ]:
def make_background_sequence(n_warmup=12, n_motion=30):
    base = make_texture(144, 208, seed=19)
    frames, masks = [], []
    for _ in range(n_warmup):
        frames.append(base.copy())
        masks.append(np.zeros_like(base, dtype=bool))
    for t in range(n_motion):
        image = base.copy()
        mask = np.zeros_like(base, dtype=bool)
        x, y, w, h = 8 + 5 * t, 52, 30, 34
        x = min(x, base.shape[1] - w - 2)
        image[y + 7:y + h + 7, x + 9:x + w + 9] = 75  # тень
        image[y:y + h, x:x + w] = 235
        mask[y:y + h, x:x + w] = True
        frames.append(image)
        masks.append(mask)
    return frames, masks


bg_frames, bg_gt = make_background_sequence()
show([bg_frames[0], bg_frames[20], bg_gt[20]],
     ["Фон", "Кадр с объектом и тенью", "GT объекта"],
     ["gray", "gray", "gray"])

**Задание 3** Реализуйте `mog2_masks`:

1. создайте `cv2.createBackgroundSubtractorMOG2(history=40, varThreshold=16, detectShadows=True)`;
2. применяйте модель с `learningRate=0.03`;
3. оставьте только значение `255`, исключив тени `127`;
4. выполните opening ядром `3×3`, затем closing ядром `7×7`;
5. верните сырые и очищенные маски.

In [ ]:
def mog2_masks(frames):
    # TODO: MOG2 -> foreground == 255 -> open 3x3 -> close 7x7
    model = cv2.createBackgroundSubtractorMOG2(
        history=40, varThreshold=16, detectShadows=True,
    )
    raw_masks, clean_masks = [], []
    open_kernel = np.ones((3, 3), np.uint8)
    close_kernel = np.ones((7, 7), np.uint8)
    for frame in frames:
      raw = model.apply(frame, learningRate=0.03)
      foreground = np.uint8(raw == 255) * 255
      clean = cv2.morphologyEx(foreground, cv2.MORPH_OPEN, open_kernel)
      clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, close_kernel)
      raw_masks.append(raw)
      clean_masks.append(clean > 0)
    return raw_masks, clean_masks

In [ ]:
raw_masks, clean_masks = mog2_masks(bg_frames)
index = 24
raw_rgb = np.zeros((*raw_masks[index].shape, 3), dtype=np.uint8)
raw_rgb[raw_masks[index] == 127] = (255, 210, 0)   # тень
raw_rgb[raw_masks[index] == 255] = (255, 0, 80)    # foreground

show([bg_frames[index], raw_rgb, clean_masks[index], bg_gt[index]],
     ["Кадр", "Raw: жёлтая тень / красный FG", "После морфологии", "GT"],
     ["gray", None, "gray", "gray"], figsize=(16, 4))

scores = [f1_score(pred, gt) for pred, gt in zip(clean_masks[12:], bg_gt[12:])]
print(f"Средний F1 после прогрева: {np.mean(scores):.3f}")
print("Уникальные значения raw-маски:", np.unique(raw_masks[index]))

### Скорость адаптации фона

`learningRate` задаёт, насколько быстро новые наблюдения влияют на модель фона.
Сравните медленную, среднюю и быструю адаптацию. Следите не только за F1, но и за
тем, не исчезает ли движущийся или временно остановившийся объект из маски.

In [ ]:
def evaluate_learning_rate(frames, ground_truth, learning_rate):
    model = cv2.createBackgroundSubtractorMOG2(
        history=40, varThreshold=16, detectShadows=True,
    )
    kernel = np.ones((3, 3), np.uint8)
    predictions = []
    for frame in frames:
        raw = model.apply(frame, learningRate=learning_rate)
        foreground = np.uint8(raw == 255) * 255
        foreground = cv2.morphologyEx(foreground, cv2.MORPH_OPEN, kernel)
        predictions.append(foreground > 0)
    scores = [f1_score(p, g) for p, g in zip(predictions[12:], ground_truth[12:])]
    return predictions, float(np.mean(scores))


rate_results = {}
for rate in (0.005, 0.03, 0.2):
    predictions, score = evaluate_learning_rate(bg_frames, bg_gt, rate)
    rate_results[rate] = predictions
    print(f"learningRate={rate:>5}: mean F1={score:.3f}, "
          f"foreground@24={predictions[24].mean():.3f}")

show(
    [rate_results[rate][24] for rate in (0.005, 0.03, 0.2)],
    ["lr=0.005", "lr=0.03", "lr=0.2"],
    ["gray"] * 3, figsize=(12, 3),
)

Какой режим предпочтительнее для камеры наблюдения, если человек
может остановиться? Почему единственного `learningRate`, оптимального для всех сцен,
не существует?

## 5. Стабилизация: KLT → RANSAC → траектория

Создадим 36 кадров с известной дрожью камеры. Для каждой соседней пары оценим
частичное аффинное преобразование по KLT-трекам. RANSAC должен отбросить неверные
соответствия. Здесь используем только перенос; в реальном видео из матрицы также
извлекают угол и масштаб.

**Задание 4** Реализуйте `estimate_translation_klt`. Используйте
`track_points`, затем `cv2.estimateAffinePartial2D(..., method=cv2.RANSAC,
ransacReprojThreshold=2.0)`. Если точек меньше 3 или матрица не найдена, верните
`(0, 0, 0.0)`; иначе — `(dx, dy, доля_инлаеров)`.

In [ ]:
def make_shaky_sequence(n=36):
    base = make_texture(160, 220, seed=23)
    rng = np.random.default_rng(23)
    positions = np.column_stack([
        3.5 * np.sin(np.arange(n) * 1.7) + rng.normal(0, 1.0, n),
        2.5 * np.cos(np.arange(n) * 1.3) + rng.normal(0, 0.8, n),
    ]).astype(np.float32)
    frames = [shift_image(base, float(dx), float(dy)) for dx, dy in positions]
    return frames, positions


shaky_frames, true_positions = make_shaky_sequence()

In [ ]:
def estimate_translation_klt(prev, nxt):
    # TODO: track_points -> estimateAffinePartial2D(RANSAC) -> dx, dy, inlier_ratio
    old, new, _ = track_points(prev, nxt, max_corners=160)
    if len(old):
      return 0, 0, 0
    matrix, inliners = cv2.estimateAffine2D(
        old, new, method=cv2.RANSAC, ransacReprojThreshold=2.0, confidence=0.99
    )
    if matrix is None or inliners is None:
      return 0, 0, 0
    return matrix[0, 2], matrix[1, 2], inliners.mean()

In [ ]:
increments = np.array([
    estimate_translation_klt(shaky_frames[i - 1], shaky_frames[i])[:2]
    for i in range(1, len(shaky_frames))
], dtype=np.float32)
inlier_ratios = np.array([
    estimate_translation_klt(shaky_frames[i - 1], shaky_frames[i])[2]
    for i in range(1, len(shaky_frames))
])
estimated_path = np.vstack([np.zeros(2), np.cumsum(increments, axis=0)])


def moving_average_path(path, radius=4):
    kernel = np.ones(2 * radius + 1, dtype=np.float32) / (2 * radius + 1)
    padded = np.pad(path, ((radius, radius), (0, 0)), mode="edge")
    return np.column_stack([
        np.convolve(padded[:, axis], kernel, mode="valid")
        for axis in range(2)
    ])


smooth_path = moving_average_path(estimated_path, radius=4)
correction = smooth_path - estimated_path
stabilized = [
    shift_image(frame, float(delta[0]), float(delta[1]))
    for frame, delta in zip(shaky_frames, correction)
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(estimated_path[:, 0], label="оценённая x")
axes[0].plot(smooth_path[:, 0], linewidth=3, label="сглаженная x")
axes[0].plot(estimated_path[:, 1], label="оценённая y")
axes[0].plot(smooth_path[:, 1], linewidth=3, label="сглаженная y")
axes[0].set(title="Траектория камеры", xlabel="Кадр", ylabel="Смещение, px")
axes[0].legend(ncol=2)
axes[0].grid(alpha=0.3)

before_step = np.diff(estimated_path, axis=0)
after_step = np.diff(smooth_path, axis=0)
axes[1].bar(["до", "после"], [before_step.std(), after_step.std()])
axes[1].set(title="Разброс межкадрового движения", ylabel="std, px")
plt.tight_layout()
plt.show()

show([shaky_frames[18], stabilized[18],
      cv2.absdiff(shaky_frames[17], shaky_frames[18]),
      cv2.absdiff(stabilized[17], stabilized[18])],
     ["До", "После", "Разность до", "Разность после"],
     ["gray", "gray", "magma", "magma"], figsize=(16, 4))
print(f"Средняя доля RANSAC-инлаеров: {inlier_ratios.mean():.3f}")

## 7. Дополнительные задания

### 7.1. Forward–backward-проверка KLT

После трекинга `frame1 → frame2` протяните найденные точки обратно
`frame2 → frame1`. Вычислите расстояние между исходной и вернувшейся точкой.
Отбросьте треки с ошибкой больше `0.5`, `1.0` и `2.0` px.

**Что сравнить:** число оставшихся точек, медианную ошибку потока и долю точек на
движущемся объекте. Объясните, почему `status == 1` недостаточно для контроля качества.

### 7.2. Собственные значения как оценка доверия

Для каждой KLT-точки возьмите значение `lambda_min` и ошибку относительно истинного
потока. Постройте scatter plot в логарифмической шкале и разделите точки на четыре
квартиля по `lambda_min`.

**Что сравнить:** медианную и 90-ю перцентиль ошибки в каждом квартиле. Проверьте
гипотезу: «чем больше `lambda_min`, тем точнее трек» — и найдите исключения.

### 7.3. Метрики плотного потока

Реализуйте:

- медиану EPE;
- `P90(EPE)`;
- долю пикселей с EPE больше 1 px;
- `Fl-all`: одновременно `EPE > 3 px` и относительная ошибка `> 5%`.

Посчитайте метрики отдельно для фона, объекта и полосы шириной 5 px около границы
объекта. Объясните, почему одна средняя EPE не описывает качество полностью.

### 7.4. Карта параметров Farnebäck

Переберите `winsize ∈ {9, 21, 41}` и `levels ∈ {1, 2, 4}` для малого и большого
смещений. Постройте две heatmap с EPE и отдельную таблицу времени.

**Вопросы:** какие параметры нужны для 14 px? Где большое окно ухудшает границы?
Есть ли конфигурация, одинаково лучшая по точности и времени?

### 7.5. Компоненты связности в маске MOG2

Добавьте к морфологической обработке `cv2.connectedComponentsWithStats`.
Удалите компоненты площадью меньше `20`, `100` и `300` пикселей.

**Что сравнить:** precision, recall и F1. Покажите случай, когда фильтр удаляет шум,
и случай, когда вместе с шумом исчезает небольшой полезный объект.

### 7.6. Когда остановившийся объект становится фоном

Измените генератор: объект движется 10 кадров, затем стоит 25 кадров. Для
`learningRate ∈ {0.005, 0.03, 0.1}` измерьте номер кадра, на котором обнаруженная
площадь становится меньше 20% истинной площади.

**Результат:** график `foreground area / GT area` по времени и объяснение связи
между скоростью адаптации и «врастанием» объекта в фон.

### 7.7. Радиус сглаживания и цена стабилизации

Сравните `radius ∈ {1, 4, 8}`. Для каждого варианта измерьте:

- стандартное отклонение межкадрового движения;
- максимальную величину коррекции;
- минимальную безопасную ширину кропа без пустых границ.

Выберите радиус для «плавного проезда камеры» и для «случайной мелкой дрожи».
Объясните, почему максимальная гладкость не всегда является лучшим результатом.

### 7.8. Движущийся объект против RANSAC

Добавьте в дрожащую последовательность крупный текстурный прямоугольник, движущийся
независимо от камеры. Увеличивайте его площадь, пока RANSAC не начнёт оценивать
движение объекта вместо камеры.

Затем исключите KLT-точки, попавшие в маску MOG2, и повторите оценку. Постройте
график ошибки движения камеры от доли кадра, занятой объектом.
